### Carregando Dados

In [1]:
import pandas as pd
import ast

In [2]:
features = pd.read_csv('../Results/feature_selection.csv').drop(columns=['Unnamed: 0'])
features.head()

,method,n_features,selected_features,rmse_mean,rmse_std,r2_mean,r2_std,alpha
0,RFS_forward,15,"['PROP_ESC_URBANA', 'PROP_MAT_PRETA', 'PROP_MA...",0.337976,0.008069,0.104115,0.045558,NaN
1,RFS_forward,13,"['PROP_ESC_URBANA', 'PROP_MAT_PRETA', 'PROP_MA...",0.338072,0.008906,0.103565,0.048478,NaN
2,RFS_forward,12,"['PROP_ESC_URBANA', 'PROP_MAT_PRETA', 'PROP_MA...",0.338166,0.008892,0.102998,0.049818,NaN
3,RFS_forward,14,"['PROP_ESC_URBANA', 'PROP_MAT_PRETA', 'PROP_MA...",0.338331,0.008414,0.102233,0.046463,NaN
4,RFE,11,"['PROP_ESC_URBANA', 'PROP_MAT_PRETA', 'PROP_MA...",0.338482,0.009082,0.101184,0.052727,NaN


In [3]:
df = pd.read_csv('../Datasets/Dados/VIF - Final até agora/dataset_modelo_2022.csv', sep=';')

In [4]:
df = df[ast.literal_eval(features.iloc[0]['selected_features']) + ['TAXA_CRE_INT', 'NO_MUNICIPIO', 'SG_UF']]

### One Hot Encoding

In [5]:
df = pd.get_dummies(
    df,
    columns=['SG_UF'],
    drop_first=False,
    dtype=int
)

### Train, Val, Test Split

In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X = df.drop(columns=['TAXA_CRE_INT'])
y = df['TAXA_CRE_INT']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

val_relative_size = 0.2 / (1 - 0.2)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_relative_size,
    random_state=42
)

### Normalização

In [8]:
from sklearn.preprocessing import StandardScaler

In [9]:
def scale_cols(X_train, X_val, X_test, columns):
    X_train_scaled = X_train.copy()
    X_val_scaled = X_val.copy()
    X_test_scaled = X_test.copy()

    scaler = StandardScaler()

    X_train_scaled[columns] = scaler.fit_transform(X_train_scaled[columns])
    X_val_scaled[columns] = scaler.transform(X_val_scaled[columns])
    X_test_scaled[columns] = scaler.transform(X_test_scaled[columns])

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

In [10]:
X_train_scaled, X_val_scaled, X_test_scaled, scaler = scale_cols(
    X_train=X_train,
    X_val=X_val,
    X_test=X_test,
    columns=ast.literal_eval(features.iloc[0]['selected_features'])
    )

### AutoML com AutoGluon

In [16]:
from autogluon.tabular import TabularPredictor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from tqdm import tqdm
import os
import shutil


In [13]:
def prepare_autogluon_data(X, y, target_column):
    data = X.copy()
    data[target_column] = y.values
    return data

In [18]:
def run_automl(
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test,
    target_column,
    id_column=None,
    output_path="autogluon_regression",
    time_limit=600,
    overwrite=True
):
    if overwrite and os.path.exists(output_path):
        shutil.rmtree(output_path)

    if id_column is not None:
        X_train = X_train.drop(columns=[id_column], errors="ignore")
        X_val = X_val.drop(columns=[id_column], errors="ignore")
        X_test = X_test.drop(columns=[id_column], errors="ignore")

    train_data = prepare_autogluon_data(X_train, y_train, target_column)
    val_data = prepare_autogluon_data(X_val, y_val, target_column)

    train_full = pd.concat(
        [train_data, val_data],
        ignore_index=True
    )

    test_data = prepare_autogluon_data(X_test, y_test, target_column)

    predictor = TabularPredictor(
        label=target_column,
        problem_type="regression",
        eval_metric="root_mean_squared_error",
        path=output_path
    )

    predictor.fit(
        train_data=train_full,
        time_limit=time_limit,
        presets="best_quality"
    )

    X_test_final = test_data.drop(columns=[target_column])

    predictions = predictor.predict(X_test_final)

    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    mae = mean_absolute_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)

    metrics = {
        "rmse": rmse,
        "mae": mae,
        "r2": r2
    }

    leaderboard = predictor.leaderboard(
        test_data,
        silent=True
    )

    return predictor, metrics, leaderboard, predictions

In [19]:
predictor, metrics, leaderboard, predictions = run_automl(
    X_train=X_train_scaled,
    y_train=y_train,
    X_val=X_val_scaled,
    y_val=y_val,
    X_test=X_test_scaled,
    y_test=y_test,
    target_column="TAXA_CRE_INT",
    id_column="NO_MUNICIPIO",
    output_path="autogluon_regression",
    time_limit=3600
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Wed May 27 14:05:53 UTC 2026
CPU Count:          16
Pytorch Version:    2.9.1+cu128
CUDA Version:       CUDA is not available
Memory Avail:       7.15 GB / 15.23 GB (46.9%)
Disk Space Avail:   59.58 GB / 182.28 GB (32.7%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `nu

(raylet) Task _ray_fit failed. There are 2 retries remaining, so the task will be retried. Error: 


	-0.3198	 = Validation score   (-root_mean_squared_error)
	1.25s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: NeuralNetTorch_BAG_L1 ... Training model for up to 1122.22s of the 1122.21s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=0.01%)
	-0.3424	 = Validation score   (-root_mean_squared_error)
	6.27s	 = Training   runtime
	0.06s	 = Validation runtime
Fitting model: LightGBMLarge_BAG_L1 ... Training model for up to 1114.77s of the 1114.76s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with ParallelLocalFoldFittingStrategy (8 workers, per: cpus=2, gpus=0, memory=1.22%)
	-0.3269	 = Validation score   (-root_mean_squared_error)
	6.21s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: CatBoost_r177_BAG_L1 ... Training model for up to 1107.15s of the 1107.14s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with Paral

In [20]:
leaderboard

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_r69_BAG_L1,-0.305733,-0.310358,root_mean_squared_error,0.010232,0.007800,1.990010,0.010232,0.007800,1.990010,1,True,33
1,NeuralNetFastAI_r160_BAG_L1,-0.306288,-0.307994,root_mean_squared_error,0.061265,0.118421,4.706770,0.061265,0.118421,4.706770,1,True,75
2,LightGBM_r96_BAG_L1,-0.306805,-0.309427,root_mean_squared_error,0.031769,0.033026,1.331033,0.031769,0.033026,1.331033,1,True,15
3,XGBoost_r89_BAG_L1,-0.306846,-0.313833,root_mean_squared_error,0.041011,0.024069,0.758708,0.041011,0.024069,0.758708,1,True,25
4,CatBoost_r60_BAG_L1,-0.306888,-0.310666,root_mean_squared_error,0.010649,0.007304,1.665435,0.010649,0.007304,1.665435,1,True,76
...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,NeuralNetTorch_r31_BAG_L1,-0.337604,-0.332468,root_mean_squared_error,0.045413,0.052477,5.877593,0.045413,0.052477,5.877593,1,True,61
103,NeuralNetTorch_r76_BAG_L1,-0.339503,-0.329801,root_mean_squared_error,0.040082,0.063499,4.126857,0.040082,0.063499,4.126857,1,True,86
104,NeuralNetTorch_r19_BAG_L1,-0.340220,-0.331017,root_mean_squared_error,0.077166,0.065811,3.972404,0.077166,0.065811,3.972404,1,True,101
105,NeuralNetTorch_r89_BAG_L1,-0.341080,-0.336418,root_mean_squared_error,0.094904,0.144104,8.368266,0.094904,0.144104,8.368266,1,True,106


In [21]:
metrics

{'rmse': np.float64(0.30747117806625385),
 'mae': 0.23923081665285262,
 'r2': 0.26678992693168113}

In [23]:
leaderboard.head()

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost_r69_BAG_L1,-0.305733,-0.310358,root_mean_squared_error,0.010232,0.007800,1.990010,0.010232,0.007800,1.990010,1,True,33
1,NeuralNetFastAI_r160_BAG_L1,-0.306288,-0.307994,root_mean_squared_error,0.061265,0.118421,4.706770,0.061265,0.118421,4.706770,1,True,75
2,LightGBM_r96_BAG_L1,-0.306805,-0.309427,root_mean_squared_error,0.031769,0.033026,1.331033,0.031769,0.033026,1.331033,1,True,15
3,XGBoost_r89_BAG_L1,-0.306846,-0.313833,root_mean_squared_error,0.041011,0.024069,0.758708,0.041011,0.024069,0.758708,1,True,25
4,CatBoost_r60_BAG_L1,-0.306888,-0.310666,root_mean_squared_error,0.010649,0.007304,1.665435,0.010649,0.007304,1.665435,1,True,76
